In [7]:
# Importation des bibliothèques nécessaires
import pandas as pd
import json
import os
import psycopg2

import utils_functions
import boto3

from psycopg2.extras import Json
from psycopg2.extras import execute_values
import math
from datetime import datetime
from io import BytesIO

In [8]:
def create_s3_client():
    """
    Crée un client S3 en utilisant les variables d'environnement pour les identifiants.
    
    :return: Client S3 Boto3
    """
    try:
        # Récupère les identifiants depuis les variables d'environnement
        access_key = os.getenv('SCW_ACCESS_KEY')
        secret_key = os.getenv('SCW_SECRET_KEY')
        endpoint_url = os.getenv('SCW_OBJECT_STORAGE_ENDPOINT')
        region = os.getenv('SCW_REGION')

        if not access_key or not secret_key or not endpoint_url or not region:
            raise ValueError("Identifiants S3 manquants dans les variables d'environnement.")

        # Initialise le client S3
        session = boto3.Session(
            aws_access_key_id=access_key,
            aws_secret_access_key=secret_key,
            region_name=region
        )
        s3_client = session.client('s3', endpoint_url=endpoint_url)

        print("✔️ Client S3 créé avec succès.")
        return s3_client
    except Exception as e:
        print(f"❌ Échec de la création du client S3 : {e}")
        raise


def charger_sheet_from_dataframe(dataframe):
    """
    Convertit un DataFrame Pandas en format JSON.
    
    :param dataframe: DataFrame Pandas
    :return: Données JSON (liste de dictionnaires)
    """
    try:
        json_data = dataframe.to_dict(orient='records')
        print(f"✔️ {len(json_data)} lignes converties en JSON.")
        return json_data
    except Exception as e:
        print(f"❌ Erreur lors de la conversion du DataFrame en JSON : {e}")
        raise


def get_connexion():
    """
    Établit une connexion PostgreSQL en utilisant les variables d'environnement.
    
    :return: Objet de connexion PostgreSQL
    """
    try:
        conn = psycopg2.connect(
            dbname=os.getenv('PG_DB_NAME'),
            user=os.getenv('PG_DB_USER'),
            password=os.getenv('PG_DB_PWD'),
            host="localhost",
            port="5432"
        )
        print("✔️ Connexion à PostgreSQL réussie.")
        return conn
    except Exception as e:
        print(f"❌ Échec de la connexion à PostgreSQL : {e}")
        raise


def create_bronze_table(conn, table_name):
    """
    Crée une table 'bronze' dans PostgreSQL si elle n'existe pas déjà.

    :param conn: Objet de connexion PostgreSQL
    :param table_name: Nom de la table à créer
    """
    try:
        cursor = conn.cursor()
        create_table_query = f"""
        CREATE TABLE IF NOT EXISTS {table_name} (
            id SERIAL PRIMARY KEY,
            data JSONB NOT NULL,
            created_at TIMESTAMPTZ DEFAULT CURRENT_TIMESTAMP
        );
        """
        cursor.execute(create_table_query)
        conn.commit()
        print(f"✔️ La table '{table_name}' est prête.")
    except Exception as e:
        print(f"❌ Erreur lors de la création de la table '{table_name}' : {e}")
        conn.rollback()
        raise
    finally:
        cursor.close()


def clean_json(obj):
    """
    Nettoie récursivement les données JSON en supprimant les valeurs invalides (ex : NaN, INF, chaînes vides).
    
    :param obj: Objet JSON
    :return: Objet JSON nettoyé
    """
    if isinstance(obj, dict):
        return {k: clean_json(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [clean_json(v) for v in obj]
    elif isinstance(obj, float):
        return None if math.isinf(obj) or math.isnan(obj) else obj
    elif isinstance(obj, str):
        return None if obj.upper() in ("INF", "NA", "NAN", "") else obj
    return obj


def import_data_to_bronze_table(conn, table_name, json_data, created_at=None):
    """
    Insère des données JSON dans une table PostgreSQL.

    :param conn: Objet de connexion PostgreSQL
    :param table_name: Nom de la table
    :param json_data: Liste de dictionnaires représentant les données JSON
    :param created_at: Horodatage de l'insertion des données (par défaut, maintenant)
    """
    try:
        cursor = conn.cursor()
        created_at = created_at or datetime.timezone.utc()

        insert_query = f"INSERT INTO {table_name} (created_at, data) VALUES %s"
        json_records = [(created_at, Json(clean_json(record))) for record in json_data]

        execute_values(cursor, insert_query, json_records)
        conn.commit()

        print(f"✔️ {len(json_records)} lignes insérées dans '{table_name}'.")
    except Exception as e:
        print(f"❌ Erreur lors de l'insertion des données dans '{table_name}' : {e}")
        conn.rollback()
        raise
    finally:
        cursor.close()


In [9]:
# Capture l'heure de traitement pour l'horodatage des données
created_at = datetime.now()
print(f"\n📅 Début du traitement à {created_at}")

# Nom du fichier à traiter et du bucket S3
file_path = "resultats_rpls_2024_v3.xlsx"
bucket_name = os.getenv('SCW_BUCKET_NAME')

# Récupération du fichier Excel depuis S3 sous forme de flux
print(f"\n🚀Chargement du fichier depuis S3 : {file_path} (Bucket: {bucket_name})")
s3_client=create_s3_client()
file_stream = utils_functions.read_excel_from_s3(s3_client, bucket_name, file_path)
print(f"✔️ Fichier chargé")

# Chargement des différentes feuilles du fichier Excel dans des DataFrames pandas
print(f"\n🚀Chargement des différentes feuilles du fichier Excel dans des DataFrames pandas")
df_region = pd.read_excel(file_stream, sheet_name="REGION", header=5)
df_departement = pd.read_excel(file_stream, sheet_name="DEPARTEMENT", header=5)
df_commune = pd.read_excel(file_stream, sheet_name="COMMUNES", header=5)
df_epci = pd.read_excel(file_stream, sheet_name="EPCI", header=5)
print(f"✔️ Feuilles chargées")

# Conversion des DataFrames en JSON pour insertion dans PostgreSQL
print("\n🚀Conversion des DataFrames en JSON")
json_data_region = charger_sheet_from_dataframe(df_region)
json_data_departement = charger_sheet_from_dataframe(df_departement)
json_data_communes = charger_sheet_from_dataframe(df_commune)
json_data_epci = charger_sheet_from_dataframe(df_epci)

# Connexion à PostgreSQL
print("\n🚀 Connexion à PostgreSQL")
conn = get_connexion()

# Définition des noms de tables dans PostgreSQL
bronze_table_name_region = "bronze.logement_rpls_region"
bronze_table_name_departement = "bronze.logement_rpls_departement"
bronze_table_name_communes = "bronze.logement_rpls_communes"
bronze_table_name_epci = "bronze.logement_rpls_epci"

# Création des tables bronze si elles n'existent pas encore
print("\n🚀Vérification et création des tables Bronze si nécessaire")
create_bronze_table (conn, bronze_table_name_region)
create_bronze_table (conn, bronze_table_name_departement)
create_bronze_table (conn, bronze_table_name_communes)
create_bronze_table (conn, bronze_table_name_epci)
print("✔️ Tables Bronze prêtes")

# Insertion des données JSON dans les tables Bronze
print("\n🚀 Insertion des données dans les tables Bronze")
import_data_to_bronze_table (conn, bronze_table_name_region, json_data_region, created_at)
import_data_to_bronze_table (conn, bronze_table_name_departement, json_data_departement, created_at)
import_data_to_bronze_table (conn, bronze_table_name_communes, json_data_communes, created_at)
import_data_to_bronze_table (conn, bronze_table_name_epci, json_data_epci, created_at)
print("✔️ Données insérées avec succès")

# Fermeture de la connexion PostgreSQL
conn.close()
print("\n✅ Fin du traitement.")



📅 Début du traitement à 2025-03-27 10:40:16.280892

🚀Chargement du fichier depuis S3 : resultats_rpls_2024_v3.xlsx (Bucket: odis-s3)
✔️ Client S3 créé avec succès.
✔️ Fichier chargé

🚀Chargement des différentes feuilles du fichier Excel dans des DataFrames pandas
✔️ Feuilles chargées

🚀Conversion des DataFrames en JSON
✔️ 23 lignes converties en JSON.
✔️ 96 lignes converties en JSON.
✔️ 16859 lignes converties en JSON.
✔️ 1325 lignes converties en JSON.

🚀 Connexion à PostgreSQL
✔️ Connexion à PostgreSQL réussie.

🚀Vérification et création des tables Bronze si nécessaire
✔️ La table 'bronze.logement_rpls_region' est prête.
✔️ La table 'bronze.logement_rpls_departement' est prête.
✔️ La table 'bronze.logement_rpls_communes' est prête.
✔️ La table 'bronze.logement_rpls_epci' est prête.
✔️ Tables Bronze prêtes

🚀 Insertion des données dans les tables Bronze
✔️ 23 lignes insérées dans 'bronze.logement_rpls_region'.
✔️ 96 lignes insérées dans 'bronze.logement_rpls_departement'.
✔️ 16859 li